In [ ]:
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from skimage.graph import route_through_array
from matplotlib.collections import LineCollection
from pathlib import Path
import copy
from scipy.ndimage import convolve
from skimage.morphology import binary_dilation


DATA_DIR = Path("/Users/sylvi/topo_data/temp/")
IMAGE_FILE = "20251209_TAF_pICOz_MQ_50_percent_damage.0_00010.topostats"
FILE_PATH = DATA_DIR / IMAGE_FILE
GRAIN_IDX = "1"


with h5py.File(FILE_PATH, "r") as f:
    image = f["image"][:]
    plt.imshow(image, cmap="afmhot", origin="lower")
    plt.show()
    print(f.keys())
    graincrops = f["grain_crops"]

    for grain_index, graincrop in graincrops.items():
        print(f"grain index: {grain_index}")
        # print(graincrop.keys())
        disordered_trace = graincrop["disordered_trace"]
        # print(disordered_trace.keys())
        disordered_trace_images = disordered_trace["images"]
        # print(disordered_trace_images.keys())
        branch_indexes_image = disordered_trace_images["branch_indexes"][:]
        branch_indexes_nonzero = np.argwhere(branch_indexes_image > 0)
        # print(branch_indexes_nonzero)
        pruned_skeleton = disordered_trace_images["pruned_skeleton"][:].astype(np.int32)

        # find the nodes
        convolved_skeleton = convolve(pruned_skeleton, np.ones((3, 3)))
        convolved_skeleton[pruned_skeleton == 0] = 0

        print(convolved_skeleton.shape)
        nodes = graincrop["nodes"]
        print(f"nodes: {nodes.keys()}")

        for node_index, node in nodes.items():
            print(f"node index: {node_index}")
            print(node.keys())

            node_area_skeleton = node["node_area_skeleton"][:]
            print(np.unique(node_area_skeleton))
            fig, ax = plt.subplots(1, 1, figsize=(10, 10))
            ax.imshow(node_area_skeleton, cmap="gray", origin="lower")
            plt.title(f"Node {node_index} area skeleton")
            plt.show()

            mask_node= np.where(node_area_skeleton == 3, 1, 0)
            mask_branch = np.where(node_area_skeleton == 1, 1, 0)

            mask_dilated_node = binary_dilation(mask_node, footprint=np.ones((3, 3)))
            mask_dilated_node_branch_intersections = mask_dilated_node * mask_branch
            fig, ax = plt.subplots(1, 1, figsize=(10, 10))
            ax.imshow(mask_dilated_node_branch_intersections, cmap="gray", origin="lower")
            plt.title(f"Node {node_index} dilated node-branch intersections")
            plt.show()

            branch_starts = np.argwhere(mask_dilated_node_branch_intersections == 1)
            if branch_starts.shape[0] != 4:
                print(f"Warning: expected 4 branch starts for node {node_index} but found {branch_starts.shape[0]}, skipping")
                continue


            



